In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q12/annotated/checkpoints/pre_cell_2.pickle

In [ ]:
Out.clear()

In [ ]:
%%RecordEvent
%%time
### cell 2 ###


date1 = pd.Timestamp("1994-01-01")
date2 = pd.Timestamp("1995-01-01")
sel = (
    (lineitem.L_RECEIPTDATE < date2)
    & (lineitem.L_COMMITDATE < date2)
    & (lineitem.L_SHIPDATE < date2)
    & (lineitem.L_SHIPDATE < lineitem.L_COMMITDATE)
    & (lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE)
    & (lineitem.L_RECEIPTDATE >= date1)
    & ((lineitem.L_SHIPMODE == "MAIL") | (lineitem.L_SHIPMODE == "SHIP"))
)
flineitem = lineitem[sel]
jn = flineitem.merge(orders, left_on="L_ORDERKEY", right_on="O_ORDERKEY")

def g1(x):
    return ((x == "1-URGENT") | (x == "2-HIGH")).sum()

def g2(x):
    return ((x != "1-URGENT") & (x != "2-HIGH")).sum()

total = jn.groupby("L_SHIPMODE", as_index=False)["O_ORDERPRIORITY"].agg((g1, g2))
total = total.reset_index()  # reset index to keep consistency with pandas
# skip sort when groupby does sort already
# total = total.sort_values("L_SHIPMODE")
print(total)



In [ ]:
%%RecordEvent
orig_output = Out.get(5)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q12/annotated/checkpoints/post_cell_2.pickle